# SPO (Subject-Predicate-Object) Triplet Extraction from Portal Files

This notebook extracts knowledge graph triplets from portal text files using transformer models.

## Import Libraries

In [ ]:
# Configuration settings
CLOBBER = False  # Set to True to overwrite existing JSON files
TEST = True      # Set to True to process only one file for testing

# Paths
PORTAL_FILES_DIR = "data/Multipurpose_Files/portal_files"
PAGE_METADATA_PATH = "Network_Innovation_Paper/data_products/page_metadata.RDS"
OUTPUT_DIR = "Network_Innovation_Paper/data_products/spo_graphs"

# Ensure output directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

## Configuration

In [ ]:
class REBELExtractor(TripletExtractor):
    \"\"\"REBEL: End-to-end relation extraction\"\"\"\n",
    "    \n",
    "    def __init__(self, model_name: str = \"Babelscape/rebel-large\"):\n",
    "        super().__init__(model_name)\n",
    "        \n",
    "        try:\n",
    "            self.pipeline = pipeline(\n",
    "                'text2text-generation', \n",
    "                model=model_name, \n",
    "                tokenizer=model_name,\n",
    "                device=0 if torch.cuda.is_available() else -1\n",
    "            )\n",
    "            \n",
    "        except Exception as e:\n",
    "            print(f\"Error loading REBEL model: {e}\")\n",
    "            self.pipeline = None\n",
    "    \n",
    "    def extract(self, text: str) -> List[Dict[str, Any]]:\n",
    "        \"\"\"Extract triplets using REBEL model\"\"\"\n",
    "        if not self.pipeline:\n",
    "            return []\n",
    "            \n",
    "        try:\n",
    "            generated = self.pipeline(\n",
    "                text, \n",
    "                return_tensors=True, \n",
    "                return_text=False,\n",
    "                max_length=512\n",
    "            )\n",
    "            \n",
    "            decoded = self.pipeline.tokenizer.batch_decode(\n",
    "                [generated[0][\"generated_token_ids\"]]\n",
    "            )[0]\n",
    "            \n",
    "            triplets = self._parse_rebel_output(decoded)\n",
    "            return triplets\n",
    "            \n",
    "        except Exception as e:\n",
    "            print(f\"Error during REBEL extraction: {e}\")\n",
    "            return []\n",
    "    \n",
    "    def _parse_rebel_output(self, output: str) -> List[Dict[str, Any]]:\n",
    "        \"\"\"Parse REBEL model output into structured triplets\"\"\"\n",
    "        triplets = []\n",
    "        \n",
    "        text = output.replace(\"<s>\", \"\").replace(\"<pad>\", \"\").replace(\"</s>\", \"\").strip()\n",
    "        \n",
    "        current_triplet = {}\n",
    "        tokens = text.split()\n",
    "        \n",
    "        i = 0\n",
    "        while i < len(tokens):\n",
    "            token = tokens[i]\n",
    "            \n",
    "            if token == \"<triplet>\":\n",
    "                if current_triplet and all(k in current_triplet for k in ['head', 'type', 'tail']):\n",
    "                    triplets.append({\n",
    "                        'subject': current_triplet['head'],\n",
    "                        'predicate': current_triplet['type'],\n",
    "                        'object': current_triplet['tail'],\n",
    "                        'confidence': 1.0\n",
    "                    })\n",
    "                current_triplet = {}\n",
    "                \n",
    "            elif token == \"<subj>\":\n",
    "                i += 1\n",
    "                subj_tokens = []\n",
    "                while i < len(tokens) and tokens[i] not in [\"<obj>\", \"<subj>\", \"<triplet>\"]:\n",
    "                    subj_tokens.append(tokens[i])\n",
    "                    i += 1\n",
    "                current_triplet['head'] = \" \".join(subj_tokens).strip()\n",
    "                i -= 1\n",
    "                \n",
    "            elif token == \"<obj>\":\n",
    "                i += 1\n",
    "                obj_tokens = []\n",
    "                while i < len(tokens) and tokens[i] not in [\"<subj>\", \"<obj>\", \"<triplet>\"]:\n",
    "                    obj_tokens.append(tokens[i])\n",
    "                    i += 1\n",
    "                current_triplet['tail'] = \" \".join(obj_tokens).strip()\n",
    "                i -= 1\n",
    "                \n",
    "            else:\n",
    "                if not current_triplet.get('type') and token not in [\"<triplet>\", \"<subj>\", \"<obj>\"]:\n",
    "                    current_triplet['type'] = token\n",
    "            \n",
    "            i += 1\n",
    "        \n",
    "        if current_triplet and all(k in current_triplet for k in ['head', 'type', 'tail']):\n",
    "            triplets.append({\n",
    "                'subject': current_triplet['head'],\n",
    "                'predicate': current_triplet['type'], \n",
    "                'object': current_triplet['tail'],\n",
    "                'confidence': 1.0\n",
    "            })\n",
    "        \n",
    "        return triplets

In [ ]:
class TripletExtractor:
    """Base class for triplet extraction models"""
    
    def __init__(self, model_name: str):
        self.model_name = model_name
        self.model = None
        self.tokenizer = None
        
    def extract(self, text: str) -> List[Dict[str, Any]]:
        """Extract triplets from raw text"""
        raise NotImplementedError

def get_files_to_process(portal_files_dir: str, output_dir: str, clobber: bool = False, test: bool = False) -> List[str]:
    \"\"\"Get list of .txt files to process based on CLOBBER and TEST settings\"\"\"
    txt_files = glob.glob(os.path.join(portal_files_dir, \"*.txt\"))
    
    if not txt_files:
        print(f\"No .txt files found in {portal_files_dir}\")
        return []
    
    files_to_process = []
    
    for txt_file in txt_files:
        base_name = os.path.splitext(os.path.basename(txt_file))[0]
        json_file = os.path.join(output_dir, f\"{base_name}.json\")
        
        if clobber or not os.path.exists(json_file):
            files_to_process.append(txt_file)
        else:
            print(f\"Skipping {txt_file} (JSON already exists)\")
    
    print(f\"Found {len(files_to_process)} files to process\")
    
    if test and files_to_process:
        print(f\"TEST mode: processing only {files_to_process[0]}\")
        return [files_to_process[0]]
    
    return files_to_process

In [ ]:
class REBELExtractor(TripletExtractor):
    """REBEL: End-to-end relation extraction"""
    
    def __init__(self, model_name: str = "Babelscape/rebel-large"):
        super().__init__(model_name)
        
        try:
            self.pipeline = pipeline(
                'text2text-generation', 
                model=model_name, 
                tokenizer=model_name,
                device=0 if torch.cuda.is_available() else -1
            )
            
        except Exception as e:
            print(f"Error loading REBEL model: {e}")
            self.pipeline = None
    
    def extract(self, text: str) -> List[Dict[str, Any]]:
        """Extract triplets using REBEL model"""
        if not self.pipeline:
            return []
            
        try:
            generated = self.pipeline(
                text, 
                return_tensors=True, 
                return_text=False,
                max_length=512
            )
            
            decoded = self.pipeline.tokenizer.batch_decode(
                [generated[0]["generated_token_ids"]]
            )[0]
            
            triplets = self._parse_rebel_output(decoded)
            return triplets
            
        except Exception as e:
            print(f"Error during REBEL extraction: {e}")
            return []
    
    def _parse_rebel_output(self, output: str) -> List[Dict[str, Any]]:
        """Parse REBEL model output into structured triplets"""
        triplets = []
        
        text = output.replace("<s>", "").replace("<pad>", "").replace("</s>", "").strip()
        
        current_triplet = {}
        tokens = text.split()
        
        i = 0
        while i < len(tokens):
            token = tokens[i]
            
            if token == "<triplet>":
                if current_triplet and all(k in current_triplet for k in ['head', 'type', 'tail']):
                    triplets.append({
                        'subject': current_triplet['head'],
                        'predicate': current_triplet['type'],
                        'object': current_triplet['tail'],
                        'confidence': 1.0
                    })
                current_triplet = {}
                
            elif token == "<subj>":
                i += 1
                subj_tokens = []
                while i < len(tokens) and tokens[i] not in ["<obj>", "<subj>", "<triplet>"]:
                    subj_tokens.append(tokens[i])
                    i += 1
                current_triplet['head'] = " ".join(subj_tokens).strip()
                i -= 1
                
            elif token == "<obj>":
                i += 1
                obj_tokens = []
                while i < len(tokens) and tokens[i] not in ["<subj>", "<obj>", "<triplet>"]:
                    obj_tokens.append(tokens[i])
                    i += 1
                current_triplet['tail'] = " ".join(obj_tokens).strip()
                i -= 1
                
            else:
                if not current_triplet.get('type') and token not in ["<triplet>", "<subj>", "<obj>"]:
                    current_triplet['type'] = token
            
            i += 1
        
        if current_triplet and all(k in current_triplet for k in ['head', 'type', 'tail']):
            triplets.append({
                'subject': current_triplet['head'],
                'predicate': current_triplet['type'], 
                'object': current_triplet['tail'],
                'confidence': 1.0
            })
        
        return triplets

## Triplex Extractor

In [ ]:
class TriplexExtractor(TripletExtractor):
    """Triplex: Specialized model for knowledge graph construction"""
    
    def __init__(self, model_name: str = "sciphi/triplex"):
        super().__init__(model_name)
        
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        
        try:
            self.model = AutoModelForCausalLM.from_pretrained(
                model_name, 
                trust_remote_code=True
            ).to(self.device).eval()
            
            self.tokenizer = AutoTokenizer.from_pretrained(
                model_name, 
                trust_remote_code=True
            )
            
        except Exception as e:
            print(f"Error loading Triplex model: {e}")
    
    def extract(self, text: str, entity_types: List[str] = None, predicates: List[str] = None) -> List[Dict[str, Any]]:
        """Extract triplets with customizable entity types and predicates"""
        if not self.model:
            return []
            
        if entity_types is None:
            entity_types = [
                "PERSON", "ORGANIZATION", "LOCATION", "DATE", 
                "PRODUCT", "EVENT", "COUNTRY", "CITY"
            ]
            
        if predicates is None:
            predicates = [
                "BORN_IN", "WORKS_FOR", "LOCATED_IN", "FOUNDED_BY", 
                "CEO_OF", "PART_OF", "HAPPENED_IN", "CREATED_BY"
            ]
        
        input_format = """Perform Named Entity Recognition (NER) and extract knowledge graph triplets from the text.
NER identifies named entities of given entity types, and triple extraction identifies relationships between entities using specified predicates.

**Entity Types:** {entity_types}
**Predicates:** {predicates}
**Text:** {text}
"""
        
        message = input_format.format(
            entity_types=json.dumps({"entity_types": entity_types}),
            predicates=json.dumps({"predicates": predicates}),
            text=text
        )
        
        messages = [{'role': 'user', 'content': message}]
        
        try:
            input_ids = self.tokenizer.apply_chat_template(
                messages, 
                add_generation_prompt=True, 
                return_tensors="pt"
            ).to(self.device)
            
            with torch.no_grad():
                output = self.model.generate(
                    input_ids=input_ids, 
                    max_length=2048,
                    temperature=0.1,
                    do_sample=True,
                    pad_token_id=self.tokenizer.eos_token_id
                )
            
            response = self.tokenizer.decode(output[0], skip_special_tokens=True)
            triplets = self._parse_triplex_output(response)
            return triplets
            
        except Exception as e:
            print(f"Error during extraction: {e}")
            return []
    
    def _parse_triplex_output(self, output: str) -> List[Dict[str, Any]]:
        """Parse Triplex model output into structured triplets"""
        triplets = []
        
        try:
            json_match = re.search(r'\[.*\]', output, re.DOTALL)
            if json_match:
                json_str = json_match.group(0)
                parsed = json.loads(json_str)
                
                for item in parsed:
                    if isinstance(item, dict) and all(key in item for key in ['subject', 'predicate', 'object']):
                        triplets.append({
                            'subject': item['subject'],
                            'predicate': item['predicate'], 
                            'object': item['object'],
                            'confidence': item.get('confidence', 1.0)
                        })
            
        except json.JSONDecodeError:
            lines = output.split('\n')
            for line in lines:
                if '->' in line or '|' in line:
                    parts = re.split(r'[->|]+', line.strip())
                    if len(parts) >= 3:
                        triplets.append({
                            'subject': parts[0].strip(),
                            'predicate': parts[1].strip(),
                            'object': parts[2].strip(),
                            'confidence': 1.0
                        })
        
        return triplets

## Utility Functions

In [ ]:
extractor = REBELExtractor()

if extractor.pipeline is not None:
    print(\"REBEL extractor ready\")
else:
    print(\"Failed to initialize REBEL extractor\")

In [ ]:
if metadata_df.empty or not files_to_process or extractor.pipeline is None:
    print(\"Cannot proceed - missing metadata, files, or extractor\")
else:
    total_triplets = 0
    
    for i, file_path in enumerate(files_to_process, 1):
        print(f\"[{i}/{len(files_to_process)}] Processing {os.path.basename(file_path)}...\")
        
        result = process_portal_file(file_path, metadata_df, extractor)
        
        file_name = result['file_name']
        base_name = os.path.splitext(file_name)[0]
        output_path = os.path.join(OUTPUT_DIR, f\"{base_name}.json\")
        
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(result, f, indent=2, ensure_ascii=False)
        
        total_triplets += result.get('total_triplets', 0)
    
    print(f\"\\nProcessed {len(files_to_process)} files\")
    print(f\"Total triplets extracted: {total_triplets}\")

In [ ]:
def process_portal_file(file_path: str, metadata_df: pd.DataFrame, extractor) -> Dict[str, Any]:
    """Process a single portal file, extracting triplets from relevant pages"""
    file_name = os.path.basename(file_path)
    
    print(f"Processing {file_name}...")
    
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            content = f.read()
        
        pages = content.split("<<PAGE_BREAK>>")
        print(f"Found {len(pages)} pages in {file_name}")
        
        file_metadata = metadata_df[metadata_df['file_name'] == file_name]
        
        if file_metadata.empty:
            print(f"No metadata found for {file_name}")
            return {
                "file_name": file_name,
                "processed_pages": 0,
                "total_pages": len(pages),
                "triplets": [],
                "error": "No metadata found"
            }
        
        relevant_pages = set(file_metadata['page_num'].tolist())
        print(f"Processing {len(relevant_pages)} relevant pages: {sorted(relevant_pages)}")
        
        all_triplets = []
        processed_pages = 0
        
        for page_num in relevant_pages:
            page_index = page_num - 1
            
            if 0 <= page_index < len(pages):
                page_text = pages[page_index].strip()
                
                if page_text:
                    print(f"  Processing page {page_num} ({len(page_text)} characters)")
                    
                    try:
                        page_triplets = extractor.extract(page_text)
                        
                        for triplet in page_triplets:
                            triplet['page_num'] = page_num
                            triplet['file_name'] = file_name
                        
                        all_triplets.extend(page_triplets)
                        processed_pages += 1
                        
                        print(f"    Extracted {len(page_triplets)} triplets from page {page_num}")
                        
                    except Exception as e:
                        print(f"    Error processing page {page_num}: {e}")
                else:
                    print(f"    Skipping empty page {page_num}")
            else:
                print(f"    Page {page_num} not found in file (only {len(pages)} pages available)")
        
        result = {
            "file_name": file_name,
            "processed_pages": processed_pages,
            "total_pages": len(pages),
            "relevant_pages": sorted(relevant_pages),
            "total_triplets": len(all_triplets),
            "triplets": all_triplets,
            "processing_timestamp": pd.Timestamp.now().isoformat()
        }
        
        print(f"Completed {file_name}: {len(all_triplets)} total triplets from {processed_pages} pages")
        return result
        
    except Exception as e:
        print(f"Error processing {file_name}: {e}")
        return {
            "file_name": file_name,
            "processed_pages": 0,
            "total_pages": 0,
            "triplets": [],
            "error": str(e),
            "processing_timestamp": pd.Timestamp.now().isoformat()
        }

## Load Page Metadata

In [ ]:
metadata_df = load_page_metadata(PAGE_METADATA_PATH)

if metadata_df.empty:
    print("Error: Could not load page metadata. Cannot continue.")
else:
    print(f"Loaded {len(metadata_df)} relevant pages")

## Get Files to Process

In [ ]:
files_to_process = get_files_to_process(
    PORTAL_FILES_DIR, 
    OUTPUT_DIR, 
    clobber=CLOBBER, 
    test=TEST
)

if not files_to_process:
    print("No files to process.")
else:
    print(f"Will process {len(files_to_process)} files")

## Initialize Extractor

In [ ]:
extractor = None

if EXTRACTOR_TYPE.lower() == "rebel":
    extractor = REBELExtractor()
    if not extractor.pipeline:
        extractor = None
elif EXTRACTOR_TYPE.lower() == "triplex":
    extractor = TriplexExtractor()
    if not extractor.model:
        extractor = None

if extractor is not None:
    print("Extractor ready")

## Process Files

In [ ]:
if metadata_df.empty or not files_to_process or extractor is None:
    print("Cannot proceed - missing metadata, files, or extractor")
else:
    total_triplets = 0
    
    for i, file_path in enumerate(files_to_process, 1):
        print(f"[{i}/{len(files_to_process)}] Processing {os.path.basename(file_path)}...")
        
        result = process_portal_file(file_path, metadata_df, extractor)
        
        file_name = result['file_name']
        base_name = os.path.splitext(file_name)[0]
        output_path = os.path.join(OUTPUT_DIR, f"{base_name}.json")
        
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(result, f, indent=2, ensure_ascii=False)
        
        total_triplets += result.get('total_triplets', 0)
    
    print(f"\nProcessed {len(files_to_process)} files")
    print(f"Total triplets extracted: {total_triplets}")

## Analyze Results

In [ ]:
# Analyze the extracted results
json_files = glob.glob(os.path.join(OUTPUT_DIR, "*.json"))

if json_files:
    print(f"Analyzing {len(json_files)} result files...")
    print("="*50)
    
    total_triplets = 0
    total_files = 0
    total_pages = 0
    
    all_subjects = []
    all_predicates = []
    all_objects = []
    
    for json_file in json_files:
        try:
            with open(json_file, 'r', encoding='utf-8') as f:
                result = json.load(f)
            
            file_name = result.get('file_name', 'Unknown')
            triplet_count = result.get('total_triplets', 0)
            page_count = result.get('processed_pages', 0)
            
            print(f"{file_name}: {triplet_count} triplets from {page_count} pages")
            
            total_triplets += triplet_count
            total_files += 1
            total_pages += page_count
            
            for triplet in result.get('triplets', []):
                all_subjects.append(triplet.get('subject', ''))
                all_predicates.append(triplet.get('predicate', ''))
                all_objects.append(triplet.get('object', ''))
                
        except Exception as e:
            print(f"Error reading {json_file}: {e}")
    
    print("="*50)
    print("SUMMARY STATISTICS")
    print("="*50)
    if total_files > 0:
        print(f"Total files processed: {total_files}")
        print(f"Total pages processed: {total_pages}")
        print(f"Total triplets extracted: {total_triplets}")
        print(f"Average triplets per file: {total_triplets/total_files:.1f}")
        if total_pages > 0:
            print(f"Average triplets per page: {total_triplets/total_pages:.1f}")
else:
    print(f"No JSON files found in {OUTPUT_DIR}")

In [ ]:
# Show most common predicates and subjects
if 'all_predicates' in locals() and all_predicates:
    from collections import Counter
    predicate_counts = Counter(all_predicates)
    print(f"\nMost common predicates:")
    for predicate, count in predicate_counts.most_common(10):
        print(f"  {predicate}: {count}")

if 'all_subjects' in locals() and all_subjects:
    subject_counts = Counter(all_subjects)
    print(f"\nMost common subjects:")
    for subject, count in subject_counts.most_common(10):
        print(f"  {subject}: {count}")

In [ ]:
# Load results into a DataFrame for further analysis
all_triplets = []

for json_file in glob.glob(os.path.join(OUTPUT_DIR, "*.json")):
    try:
        with open(json_file, 'r', encoding='utf-8') as f:
            result = json.load(f)
        
        for triplet in result.get('triplets', []):
            all_triplets.append(triplet)
            
    except Exception as e:
        print(f"Error reading {json_file}: {e}")

if all_triplets:
    triplets_df = pd.DataFrame(all_triplets)
    print(f"\nLoaded {len(triplets_df)} triplets into DataFrame")
    print("\nSample triplets:")
    print(triplets_df[['subject', 'predicate', 'object']].head())
    
    # Example: Filter triplets by predicate containing sustainability terms
    if 'predicate' in triplets_df.columns:
        sustainability_triplets = triplets_df[
            triplets_df['predicate'].str.contains('sustain|environment|green|eco', case=False, na=False)
        ]
        print(f"\nFound {len(sustainability_triplets)} sustainability-related triplets")
else:
    print("\nNo triplets found to load into DataFrame")